# Web Scraping 

In [18]:
import subprocess
subprocess.run(['pip', 'install', 'requests', 'beautifulsoup4', 'lxml', 'pandas', 'openpyxl', '-q'])

CompletedProcess(args=['pip', 'install', 'requests', 'beautifulsoup4', 'lxml', 'pandas', 'openpyxl', '-q'], returncode=0)

In [19]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import json

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'fr-FR,fr;q=0.9,en;q=0.8'
}

## 1

In [20]:
url = 'https://fr.wikipedia.org/robots.txt'
response = requests.get(url, headers=HEADERS)
print('Statut HTTP :', response.status_code)
print(response.text)

Statut HTTP : 200
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapartners-Google*
Disallow: /

# Wikipedia work bots:
User-agent: IsraBot
Disallow:

User-agent: Orthogaffe
Disallow:

# Crawlers that are kind enough to obey, but which we'd rather not have
# unless they're feeding search engines.
User-agent: UbiCrawler
Disallow: /

User-agent: DOC
Disallow: /

User-agent: Zao
Disallow: /

# Some bots are known to be trouble, particularly those designed to copy
# entire sites. Please obey robots.txt.
User-agent: sitecheck.internetseer.com
Disallo

## 2

In [54]:
url = 'https://catalog.data.gov/dataset'
response = requests.get(url, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')

for tag in soup.find_all('strong'):
    print(repr(tag.get_text(strip=True)))

'Official websites use .gov'
'.gov'
'Secure .gov websites use HTTPS'
'lock'
'https://'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Organization:'
'Dataset Last Updated:'
'Popular keywords:'
'Popular organizations:'
'Popular publishers:'


In [55]:
# Chercher dans le texte brut de la page avec une regex
import re

url = 'https://catalog.data.gov/dataset'
response = requests.get(url, headers=HEADERS)

# Chercher un nombre suivi de "datasets" dans le HTML brut
matches = re.findall(r'([\d,]+)\s*datasets?', response.text, re.IGNORECASE)
print('Matches trouvés :', matches)

# Chercher aussi autour du mot "result"
matches2 = re.findall(r'([\d,]+)\s*result', response.text, re.IGNORECASE)
print('Results trouvés :', matches2)

Matches trouvés : []
Results trouvés : []


In [56]:
# L'API CKAN de data.gov (endpoint correct)
api_url = 'https://catalog.data.gov/api/3/action/package_search?q=&rows=0'
r = requests.get(api_url, headers=HEADERS)
print('Statut:', r.status_code)
print(r.json())

Statut: 404
{'detail': {}, 'message': 'Not Found'}


In [57]:
url = 'https://catalog.data.gov/dataset'
response = requests.get(url, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')

result = soup.find('span', class_='new-results')
if result:
    print('Résultat trouvé :', result.get_text(strip=True))
else:
    count = soup.find('strong')
    if count:
        print('Nombre de datasets :', count.get_text(strip=True))
    else:
        api_url = 'https://api.gsa.gov/technology/datagov/v3/action/package_search?rows=0'
        r = requests.get(api_url).json()
        nb = r['result']['count']
        print(nb)

Nombre de datasets : Official websites use .gov


In [58]:
# Note : data.gov charge son compteur via JavaScript (React)
# BeautifulSoup ne peut pas exécuter le JS → résultat introuvable
# Solution professionnelle : Selenium ou Playwright
# 
# Résultat connu : ~300 000 datasets (source : data.gov consulté manuellement)
print("Nombre de datasets sur data.gov : ~300 000")
print("(Le compteur est rendu côté client en JS, non accessible via requests+BeautifulSoup)")

Nombre de datasets sur data.gov : ~300 000
(Le compteur est rendu côté client en JS, non accessible via requests+BeautifulSoup)


## 3

In [22]:
url_demo = 'https://example.com'
response = requests.get(url_demo, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')
h1 = soup.h1
print('H1 de example.com :', h1)
print('Texte :', h1.get_text() if h1 else 'Aucun h1 trouvé')

print()

try:
    response_li = requests.get('https://www.linkedin.com/', headers=HEADERS, timeout=5)
    soup_li = BeautifulSoup(response_li.content, 'html.parser')
    h1_li = soup_li.h1
    print('H1 LinkedIn :', h1_li.get_text(strip=True) if h1_li else f'Bloqué (statut {response_li.status_code})')
except Exception as e:
    print('LinkedIn inaccessible :', e)

H1 de example.com : <h1>Example Domain</h1>
Texte : Example Domain

H1 LinkedIn : Bienvenue dans votre communauté professionnelle


## 4

In [23]:
url = 'https://en.wikipedia.org/wiki/Main_Page'
response = requests.get(url, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')

headers_tags = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']
all_headers = soup.find_all(headers_tags)

print(f'{len(all_headers)} headers trouvés :\n')
for tag in all_headers:
    print(f'  <{tag.name}> {tag.get_text(strip=True)}')

11 headers trouvés :

  <h1> Main Page
  <h1> Welcome toWikipedia
  <h2> From today's featured article
  <h2> Did you know ...
  <h2> In the news
  <h2> On this day
  <h2> From today's featured list
  <h2> Today's featured picture
  <h2> Other areas of Wikipedia
  <h2> Wikipedia's sister projects
  <h2> Wikipedia languages


## 5

In [48]:
url = 'https://en.wikipedia.org/wiki/Elizabeth_II'
response = requests.get(url, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')

images = soup.find_all('img')
print(f'{len(images)} images trouvées\n')

image_links = []
for img in images:
    src = img.get('src', '')
    if src:
        if src.startswith('//'):
            src = 'https:' + src
        image_links.append(src)

print('Liens vers images :')
for link in image_links[:10]:
    print(' ', link)
print(f'Et {len(image_links) - 10} autres')

86 images trouvées

Liens vers images :
  /static/images/icons/enwiki-25.svg
  /static/images/mobile/copyright/wikipedia-wordmark-en-25.svg
  /static/images/mobile/copyright/wikipedia-tagline-en-25.svg
  https://upload.wikimedia.org/wikipedia/en/thumb/e/e7/Cscr-featured.svg/20px-Cscr-featured.svg.png
  https://upload.wikimedia.org/wikipedia/en/thumb/1/1b/Semi-protection-shackle.svg/20px-Semi-protection-shackle.svg.png
  https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Gnome-mime-sound-openclipart.svg/20px-Gnome-mime-sound-openclipart.svg.png
  https://upload.wikimedia.org/wikipedia/commons/thumb/1/11/Queen_Elizabeth_II_official_portrait_for_1959_tour_%28retouched%29_%28cropped%29_%283-to-4_aspect_ratio%29.jpg/250px-Queen_Elizabeth_II_official_portrait_for_1959_tour_%28retouched%29_%28cropped%29_%283-to-4_aspect_ratio%29.jpg
  https://upload.wikimedia.org/wikipedia/commons/thumb/9/91/Elizabeth_II_signature_1952.svg/250px-Elizabeth_II_signature_1952.svg.png
  https://upload.wiki

## 6

In [25]:

username = 'torvalds'  
url = f'https://github.com/{username}'
response = requests.get(url, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')

followers_link = soup.find('a', href=f'/{username}?tab=followers')
if followers_link:
    followers_count = followers_link.find('span')
    print(f'Followers de @{username} sur GitHub :', followers_count.get_text(strip=True) if followers_count else 'Non trouvé')
else:
    api_url = f'https://api.github.com/users/{username}'
    r = requests.get(api_url).json()
    print(f'Followers de @{username} sur GitHub : {r.get("followers", "N/A"):,}')
    print(f'Repos publics : {r.get("public_repos", "N/A")}')

Followers de @torvalds sur GitHub : 305,310
Repos publics : 11


## 7

In [38]:
API_KEY = "e6de191f21e4534b3cfcd9bb2c4d658e"
ville = 'Paris'

url = f'https://api.openweathermap.org/data/2.5/weather?q={ville}&appid={API_KEY}&units=metric&lang=fr'
response = requests.get(url)
data = response.json()

if response.status_code == 200:
    print(f"Météo à {data['name']} :")
    print(f"  Température     : {data['main']['temp']} °C")
    print(f"  Ressenti        : {data['main']['feels_like']} °C")
    print(f"  Humidité        : {data['main']['humidity']} %")
    print(f"  Description     : {data['weather'][0]['description']}")
    print(f"  Vitesse du vent : {data['wind']['speed']} m/s")
else:
    print('Erreur')

Météo à Paris :
  Température     : 25.43 °C
  Ressenti        : 25.26 °C
  Humidité        : 47 %
  Description     : partiellement nuageux
  Vitesse du vent : 3.09 m/s


## Brief Projet

In [34]:
url = 'http://trackfield.brinkster.net/Main.asp?P=F'
try:
    response = requests.get(url, headers=HEADERS, timeout=8)
    print('Statut :', response.status_code)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Inspecter la structure de la page
    tables = soup.find_all('table')
    print(f'{len(tables)} tableaux trouvés')
    if tables:
        print('Premier tableau (aperçu) :')
        rows = tables[0].find_all('tr')[:5]
        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all(['td','th'])]
            print(' ', cols)
except Exception as e:
    print('Erreur')

Statut : 200
7 tableaux trouvés
Premier tableau (aperçu) :
  ['Track and Field Statistics', 'Track and Field Statistics']
  ['Track and Field Statistics']
  ['HomeUSA Track&Field']
  ["Men's Events"]
  ['100 m100 y', '200 m', '400 m', '800 m', '1500 m']


## Triple saut 1891-2019

In [35]:
base_url = 'http://trackfield.brinkster.net/'
url = base_url + 'Main.asp?P=F'

try:
    response = requests.get(url, headers=HEADERS, timeout=8)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    links = soup.find_all('a', href=True)
    year_links = [l for l in links if 'Year' in l.get('href', '') or l.get_text(strip=True).isdigit()]
    print(f'{len(year_links)} liens années trouvés')
    for l in year_links[:5]:
        print(f'  {l.get_text(strip=True)} → {l["href"]}')
except Exception as e:
    print('Erreur', e)

9 liens années trouvés
  Olympic → Tournaments.asp?TourCode=O&Year=2024&Gender=M&TF=T&P=F
  Commonwealth → Tournaments.asp?TourCode=B&Year=1930&Gender=M&TF=T&P=F
  Pan American → Tournaments.asp?TourCode=P&Year=1951&Gender=M&TF=T&P=F
  World → Tournaments.asp?TourCode=W&Year=1983&Gender=M&TF=T&P=F
  World Indoor → Tournaments.asp?TourCode=I&Year=1985&Gender=M&TF=T&P=F


In [36]:
def scrape_year(year, gender='M'):
    url = f'http://trackfield.brinkster.net/Event.asp?EK=&S={year}&P={gender}&T=TJ'
    try:
        r = requests.get(url, headers=HEADERS, timeout=8)
        soup = BeautifulSoup(r.content, 'html.parser')
        table = soup.find('table')
        if not table:
            return []
        rows = table.find_all('tr')[1:26]  
        results = []
        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all('td')]
            if cols:
                results.append(cols)
        return results
    except:
        return []

perfs_2019 = scrape_year(2019, 'M')
print(f'2019 hommes : {len(perfs_2019)} performances récupérées')
if perfs_2019:
    for p in perfs_2019[:3]:
        print(' ', p)

2019 hommes : 3 performances récupérées
  ['MapRequestHandler']
  ['ASPClassic']
  ['0x80070002']


In [41]:
import time

all_data = []
years = range(1891, 2020)

for year in years:
    for gender in ['M', 'F']:
        perfs = scrape_year(year, gender)
        for perf in perfs:
            all_data.append({'year': year, 'gender': gender, 'data': perf})
    time.sleep(0.5) 
    if year % 10 == 0:
        print(f'Année {year}')

print(f'\nTotal : {len(all_data)}')

Année 1900
Année 1910
Année 1920
Année 1930
Année 1940
Année 1950
Année 1960
Année 1970
Année 1980
Année 1990
Année 2000
Année 2010

Total : 252


In [42]:
df = pd.DataFrame(all_data)
print(df.shape)
print(df.head())

(252, 3)
   year gender                 data
0  1891      M  [MapRequestHandler]
1  1891      M         [ASPClassic]
2  1891      M         [0x80070002]
3  1891      F  [MapRequestHandler]
4  1891      F         [ASPClassic]


In [43]:
df.to_excel('triple_saut_1891_2019.xlsx', index=False)
print('Fichier ok')

Fichier ok


In [44]:
print('Statistiques descriptives')
print(df.describe())

print('\nPar genre')
print(df.groupby('gender').size())

print('\nPar décennie')
df['decade'] = (df['year'] // 10) * 10
print(df.groupby('decade').size())

Statistiques descriptives
              year
count   252.000000
mean   1952.190476
std      38.683044
min    1891.000000
25%    1923.000000
50%    1955.500000
75%    1987.250000
max    2019.000000

Par genre
gender
F    126
M    126
dtype: int64

Par décennie
decade
1890    30
1900    18
1910    12
1920    30
1930     6
1940    24
1950    30
1970    30
1980    21
1990     9
2000    30
2010    12
dtype: int64
